# Notebook 3: CIFAR-10, Overfitting and Regularisation

**Goal.** Move from the "easy" MNIST to natural colour images, watch a plain CNN overfit, then fix it step by step with the standard toolkit: normalisation, data augmentation, batch normalisation, dropout, weight decay and learning-rate scheduling.

**What you will learn**

1. Generalisation, overfitting and the bias-variance picture for neural networks.
2. Per-channel normalisation and why it matters for colour images.
3. Data augmentation as a way to encode invariances.
4. Batch Normalization: the algorithm, train vs eval behaviour, and why it helps.
5. Dropout and weight decay as regularisers.
6. Learning-rate schedules (cosine annealing, one-cycle).
7. A controlled experiment: one change at a time, measured on a validation set.

> **Runtime:** use a GPU (*Runtime → Change runtime type → GPU*). The full notebook trains three models; on a T4 it takes roughly 10 to 15 minutes. Lower `EPOCHS` if you are in a hurry.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import time, copy

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. The dataset

CIFAR-10: 60,000 colour images of size $32\times32\times3$ in 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck); 50,000 for training and 10,000 for testing.

Compared with MNIST, it is much harder: objects appear at different scales, poses, colours and backgrounds, and classes like cat/dog or automobile/truck are visually close. A good MLP reaches about 55%; a small CNN about 70 to 75%; a well-regularised small CNN 85 to 90%; state-of-the-art models above 99%.

### Per-channel normalisation

We compute the mean and standard deviation of each colour channel over the training set, and normalise each channel separately:

$$
\tilde{x}_{c,i,j} = \frac{x_{c,i,j} - \mu_c}{\sigma_c}.
$$

Why per channel? The three channels have different statistics (CIFAR images are slightly blue-shifted on average: sky and sea). Normalising each gives every input dimension zero mean and unit variance, which keeps the first layer's pre-activations at a healthy scale and conditions the optimisation problem.

**Important:** statistics come from the *training* set only, and the same numbers are then applied to validation and test data. Computing them on the test set would leak information.

In [ ]:
raw = datasets.CIFAR10(root="./data", train=True, download=True, transform=transforms.ToTensor())
loader = DataLoader(raw, batch_size=1000, shuffle=False)
s, s2, n = torch.zeros(3), torch.zeros(3), 0
for x, _ in loader:
    s += x.sum(dim=(0, 2, 3)); s2 += (x ** 2).sum(dim=(0, 2, 3)); n += x.numel() // 3
MEAN = s / n
STD = (s2 / n - MEAN ** 2).sqrt()
print("mean:", MEAN.numpy().round(4), " std:", STD.numpy().round(4))
CLASSES = raw.classes
print(CLASSES)

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(14, 4.5))
idx_by_class = {c: [] for c in range(10)}
for i, t in enumerate(raw.targets):
    if len(idx_by_class[t]) < 3:
        idx_by_class[t].append(i)
for c in range(10):
    for r in range(3):
        ax = axes[r, c]
        ax.imshow(raw[idx_by_class[c][r]][0].permute(1, 2, 0))  # CHW -> HWC for matplotlib
        ax.axis("off")
        if r == 0: ax.set_title(CLASSES[c], fontsize=9)
plt.tight_layout(); plt.show()

## 2. Generalisation and overfitting

What we really care about is the **expected risk** on new data from the same distribution, $R(\theta) = \mathbb{E}_{(x,y)\sim P}[\mathcal{L}(f_\theta(x), y)]$. We can only minimise the **empirical risk** on the training set. The difference between the two is the **generalisation gap**.

A network with millions of parameters can memorise 50,000 images, labels and all (Zhang et al., 2017, showed CNNs can even fit *random* labels perfectly). When that happens, training accuracy approaches 100% while validation accuracy stalls or falls: **overfitting**.

**Regularisation** is anything that reduces the generalisation gap, usually by restricting what the model can easily fit or by injecting noise. We will add techniques one at a time and measure each on a held-out validation set:

| Experiment | Changes |
|---|---|
| A. Baseline | plain CNN, no augmentation |
| B. + Augmentation | random crops and flips |
| C. + BatchNorm, Dropout, weight decay, cosine LR | the "full" recipe |

## 3. Data augmentation

Augmentation applies random, label-preserving transformations to each training image every time it is loaded. The network effectively sees a new variant each epoch, so memorisation becomes much harder.

Conceptually, augmentation tells the model which transformations **should not** change the label, an invariance we know from domain knowledge:

* `RandomCrop(32, padding=4)`: pad by 4 pixels and crop a random $32\times32$ window. Encodes *small translations*.
* `RandomHorizontalFlip()`: mirror with probability 0.5. A mirrored cat is still a cat. (We would **not** flip MNIST digits: a mirrored "2" is not a "2". Augmentations are domain-specific.)
* Optional: `ColorJitter` (lighting), `RandomRotation`, `RandAugment`, Cutout/`RandomErasing`.

Augmentation is applied **only to training data**. Validation and test images go through the deterministic transform.

In [ ]:
plain_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN.tolist(), STD.tolist()),
])
aug_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN.tolist(), STD.tolist()),
])

# Visualise: the same image, augmented 8 times
show_tf = transforms.Compose([transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
                              transforms.RandomHorizontalFlip()])
pil_img = datasets.CIFAR10(root="./data", train=True, download=False)[7][0]
fig, axes = plt.subplots(1, 9, figsize=(13, 1.8))
axes[0].imshow(pil_img); axes[0].set_title("original", fontsize=9)
for ax in axes[1:]:
    ax.imshow(show_tf(pil_img))
for ax in axes: ax.axis("off")
plt.show()

In [ ]:
# Train/validation split: 45k / 5k. Two dataset objects so train and val can use different transforms.
perm = torch.randperm(50000, generator=torch.Generator().manual_seed(0))
train_idx, val_idx = perm[:45000].tolist(), perm[45000:].tolist()

def make_loaders(train_transform, batch_size=128):
    tr = Subset(datasets.CIFAR10("./data", train=True, download=False, transform=train_transform), train_idx)
    va = Subset(datasets.CIFAR10("./data", train=True, download=False, transform=plain_tf), val_idx)
    kw = dict(num_workers=2, pin_memory=torch.cuda.is_available())
    return (DataLoader(tr, batch_size=batch_size, shuffle=True, drop_last=True, **kw),
            DataLoader(va, batch_size=512, shuffle=False, **kw))

test_set = datasets.CIFAR10("./data", train=False, download=True, transform=plain_tf)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=2)

## 4. Batch Normalization

### 4.1 The algorithm

For a conv layer's output with shape $(N, C, H, W)$, BatchNorm normalises **each channel** $c$ using statistics over the batch and spatial positions (all $N\cdot H\cdot W$ values of that channel):

$$
\mu_c = \frac{1}{NHW}\sum_{n,i,j} x_{n,c,i,j},
\qquad
\sigma_c^2 = \frac{1}{NHW}\sum_{n,i,j} (x_{n,c,i,j} - \mu_c)^2,
$$

$$
\hat{x}_{n,c,i,j} = \frac{x_{n,c,i,j} - \mu_c}{\sqrt{\sigma_c^2 + \epsilon}},
\qquad
y_{n,c,i,j} = \gamma_c\, \hat{x}_{n,c,i,j} + \beta_c.
$$

$\gamma_c$ and $\beta_c$ are **learnable** scale and shift, so the layer can undo the normalisation if that is optimal. `nn.BatchNorm2d(C)` therefore has $2C$ parameters (plus $2C$ non-learned buffers, below).

### 4.2 Train vs eval

At test time we might predict on a single image, so batch statistics are unavailable (and would make predictions depend on other images in the batch). During training BatchNorm therefore keeps **running averages**:

$$
\mu^\text{run} \leftarrow (1-m)\,\mu^\text{run} + m\,\mu_\text{batch}, \qquad m = 0.1 \text{ (PyTorch's `momentum`)}.
$$

In `model.eval()` mode these running statistics are used instead. **Forgetting `model.eval()` is one of the most common PyTorch bugs**: predictions become noisy and depend on the batch composition.

### 4.3 Why it helps

* Allows **much higher learning rates** and makes training far less sensitive to initialisation. The original paper attributed this to reducing "internal covariate shift"; later work (Santurkar et al., 2018) argues the main effect is a **smoother loss landscape**, i.e. better-behaved gradients.
* Acts as a mild **regulariser**: each example's normalisation depends on the random other members of its mini-batch, which injects noise.

### 4.4 Practical rules

* Typical ordering: `Conv → BatchNorm → ReLU`.
* Set `bias=False` in a conv followed by BatchNorm: the mean subtraction cancels any bias, and $\beta$ plays its role.
* BatchNorm misbehaves with very small batches (say below 8). Alternatives: GroupNorm, LayerNorm.

In [ ]:
# Verify the BatchNorm formula by hand
bn = nn.BatchNorm2d(4)
x = torch.randn(8, 4, 5, 5) * 3 + 2
bn.train()
y = bn(x)
mu = x.mean(dim=(0, 2, 3), keepdim=True)
var = x.var(dim=(0, 2, 3), unbiased=False, keepdim=True)
manual = (x - mu) / torch.sqrt(var + bn.eps) * bn.weight.view(1, -1, 1, 1) + bn.bias.view(1, -1, 1, 1)
print("max diff vs manual:", (y - manual).abs().max().item())
print("per-channel mean after BN:", y.mean(dim=(0, 2, 3)).detach().numpy().round(4))
print("per-channel std  after BN:", y.std(dim=(0, 2, 3)).detach().numpy().round(3))
print("running_mean after one step:", bn.running_mean.numpy().round(3), "(0.1 * batch mean)")

# Train vs eval: the same input gives different outputs
bn.eval()
print("eval-mode output differs from train-mode:", not torch.allclose(bn(x), y))

## 5. Dropout and weight decay

### Dropout

During training, `nn.Dropout(p)` sets each activation to zero with probability $p$ and scales the survivors by $1/(1-p)$ so that the expected value is unchanged:

$$
\tilde{h}_i = \frac{m_i}{1-p}\, h_i, \qquad m_i \sim \text{Bernoulli}(1-p).
$$

At eval time it does nothing. The network cannot rely on any single unit, which discourages fragile co-adaptations; it can be viewed as training an exponentially large ensemble of sub-networks that share weights. In CNNs dropout is most useful in fully connected heads; for conv feature maps, `nn.Dropout2d` (dropping whole channels) is the more sensible variant because neighbouring pixels are strongly correlated.

### Weight decay

Weight decay adds $\frac{\lambda}{2}\|\theta\|^2$ to the loss (L2 regularisation), pulling weights toward zero and favouring "simpler" functions. With plain SGD it is equivalent to multiplying weights by $(1-\eta\lambda)$ each step. With Adam the two are **not** equivalent, which is why **AdamW** applies the decay directly to the weights ("decoupled weight decay", Loshchilov and Hutter, 2019). Typical values: `5e-4` for SGD on CIFAR, `1e-2` to `5e-2` for AdamW. Biases and BatchNorm parameters are often excluded from decay.

In [ ]:
drop = nn.Dropout(p=0.5)
h = torch.ones(1, 10)
drop.train(); print("train:", drop(h))
drop.eval();  print("eval: ", drop(h))

## 6. Learning-rate schedules

A large learning rate makes fast progress early but bounces around the minimum; a small one converges precisely but slowly. Schedules get the best of both by **decaying** the learning rate during training.

* **Step decay:** divide by 10 at fixed epochs. Classic but needs tuning.
* **Cosine annealing:** $\eta_t = \eta_\text{min} + \tfrac{1}{2}(\eta_\text{max} - \eta_\text{min})\left(1 + \cos\frac{\pi t}{T}\right)$. Smooth, one hyperparameter ($T$ = total steps).
* **One-cycle** (Smith, 2018): warm up linearly from small to large, then anneal down. Warm-up avoids instability at the very start, when gradients are large and BatchNorm statistics are poor.

We step the scheduler **every batch** (not every epoch) for a smoother curve.

In [ ]:
dummy = torch.optim.SGD([nn.Parameter(torch.zeros(1))], lr=0.1)
T = 1000
cos = torch.optim.lr_scheduler.CosineAnnealingLR(dummy, T_max=T)
lrs_cos = []
for _ in range(T): lrs_cos.append(dummy.param_groups[0]["lr"]); dummy.step(); cos.step()
dummy = torch.optim.SGD([nn.Parameter(torch.zeros(1))], lr=0.1)
one = torch.optim.lr_scheduler.OneCycleLR(dummy, max_lr=0.1, total_steps=T)
lrs_one = []
for _ in range(T): lrs_one.append(dummy.param_groups[0]["lr"]); dummy.step(); one.step()
plt.figure(figsize=(7, 3))
plt.plot(lrs_cos, label="cosine annealing"); plt.plot(lrs_one, label="one-cycle")
plt.xlabel("step"); plt.ylabel("learning rate"); plt.legend(); plt.show()

## 7. The models

The baseline is a VGG-style network: blocks of two $3\times3$ convs followed by $2\times2$ max pooling, doubling the channels at each downsampling. The improved model is identical in shape but adds BatchNorm after every conv and dropout in the head, so any difference comes from the regularisers, not the architecture.

In [ ]:
def conv_block(c_in, c_out, bn):
    layers = []
    for i in range(2):
        layers.append(nn.Conv2d(c_in if i == 0 else c_out, c_out, 3, padding=1, bias=not bn))
        if bn:
            layers.append(nn.BatchNorm2d(c_out))
        layers.append(nn.ReLU(inplace=True))
    layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)


class VGGish(nn.Module):
    def __init__(self, bn=False, dropout=0.0, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32, bn),     # 32x32 -> 16x16
            conv_block(32, 64, bn),    # 16x16 -> 8x8
            conv_block(64, 128, bn),   # 8x8   -> 4x4
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))

for bn in (False, True):
    m = VGGish(bn=bn)
    print(f"bn={bn}: {sum(p.numel() for p in m.parameters()):,} params, output {tuple(m(torch.zeros(2, 3, 32, 32)).shape)}")

## 8. A training harness

We reuse the loop from Notebook 1 with three additions: an optional per-batch scheduler, **automatic mixed precision** (AMP) on GPU for speed, and keeping the **best model by validation accuracy** (a simple form of early stopping).

AMP runs most operations in 16-bit floats, roughly doubling throughput on modern GPUs. A `GradScaler` multiplies the loss by a large factor before `backward()` so small gradients do not underflow in fp16, then un-scales them before the optimizer step.

In [ ]:
use_amp = device.type == "cuda"

def run_epoch(model, loader, loss_fn, optimizer=None, scheduler=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = loss_fn(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, epochs, optimizer, scheduler=None, label=""):
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler(enabled=use_amp)
    hist = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}
    best_acc, best_state = 0.0, None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, train_loader, loss_fn, optimizer, scheduler, scaler)
        vl, va = run_epoch(model, val_loader, loss_fn)
        for k, v in zip(hist, (tl, ta, vl, va)): hist[k].append(v)
        if va > best_acc:
            best_acc, best_state = va, copy.deepcopy(model.state_dict())
        print(f"[{label}] ep {ep:2d} | train {tl:.3f}/{ta:.3f} | val {vl:.3f}/{va:.3f} | "
              f"lr {optimizer.param_groups[0]['lr']:.2e} | {time.time() - t0:.0f}s")
    model.load_state_dict(best_state)
    return hist

## 9. Experiment A: baseline (no augmentation, no BatchNorm)

Watch the gap between training and validation accuracy open up.

In [ ]:
EPOCHS = 15
results = {}

train_loader, val_loader = make_loaders(plain_tf)
model_a = VGGish(bn=False, dropout=0.0).to(device)
opt = torch.optim.Adam(model_a.parameters(), lr=1e-3)
results["A: baseline"] = fit(model_a, train_loader, val_loader, EPOCHS, opt, label="A")

## 10. Experiment B: + data augmentation

Same model and optimizer; only the training transform changes.

In [ ]:
train_loader_aug, val_loader = make_loaders(aug_tf)
model_b = VGGish(bn=False, dropout=0.0).to(device)
opt = torch.optim.Adam(model_b.parameters(), lr=1e-3)
results["B: + augmentation"] = fit(model_b, train_loader_aug, val_loader, EPOCHS, opt, label="B")

## 11. Experiment C: the full recipe

Augmentation + BatchNorm + dropout + SGD with momentum and weight decay + one-cycle schedule. Thanks to BatchNorm we can use a much larger learning rate (0.1 peak with SGD).

SGD with momentum keeps a velocity $v \leftarrow \mu v + g$ and updates $\theta \leftarrow \theta - \eta v$. On image classification with BatchNorm it often generalises a bit better than Adam, though either works.

In [ ]:
model_c = VGGish(bn=True, dropout=0.3).to(device)

# Exclude biases and BatchNorm parameters (all 1D tensors) from weight decay
decay = [p for p in model_c.parameters() if p.ndim > 1]
no_decay = [p for p in model_c.parameters() if p.ndim <= 1]
opt = torch.optim.SGD([{"params": decay, "weight_decay": 5e-4},
                       {"params": no_decay, "weight_decay": 0.0}],
                      lr=0.1, momentum=0.9, nesterov=True)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, epochs=EPOCHS,
                                            steps_per_epoch=len(train_loader_aug))
results["C: full recipe"] = fit(model_c, train_loader_aug, val_loader, EPOCHS, opt, sched, label="C")

## 12. Comparing the experiments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (name, h), color in zip(results.items(), ["tab:red", "tab:blue", "tab:green"]):
    ep = range(1, len(h["val_acc"]) + 1)
    axes[0].plot(ep, h["train_acc"], "--", color=color, alpha=0.6)
    axes[0].plot(ep, h["val_acc"], "-", color=color, label=name)
    axes[1].plot(ep, h["train_loss"], "--", color=color, alpha=0.6)
    axes[1].plot(ep, h["val_loss"], "-", color=color, label=name)
axes[0].set_title("accuracy (dashed = train, solid = val)"); axes[0].legend()
axes[1].set_title("loss (dashed = train, solid = val)"); axes[1].legend()
for a in axes: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

print(f"{'experiment':<22}{'best val acc':>14}{'final train-val gap':>22}")
for name, h in results.items():
    print(f"{name:<22}{max(h['val_acc']):>14.4f}{h['train_acc'][-1] - h['val_acc'][-1]:>22.4f}")

**What you should see** (exact numbers vary by run):

* **A** reaches high training accuracy quickly while validation loss starts *rising* after a few epochs: textbook overfitting. Validation accuracy plateaus around 75 to 78%.
* **B** trains more slowly (the task is harder: every image is new each epoch), but the gap shrinks dramatically and validation accuracy keeps improving.
* **C** is the best of both: BatchNorm and the schedule make optimisation fast, augmentation and dropout keep the gap small. Typically 85 to 88% in 15 epochs, and above 90% with ~40 epochs.

Note that a rising validation *loss* can coincide with a still-rising validation *accuracy*: the model becomes over-confident on the examples it gets wrong, which the loss punishes heavily but accuracy ignores.

## 13. Test-set evaluation of the final model

In [ ]:
test_loss, test_acc = run_epoch(model_c, test_loader, nn.CrossEntropyLoss())
print(f"Model C test accuracy: {test_acc:.4f}")

# Per-class accuracy
model_c.eval()
correct = torch.zeros(10); total = torch.zeros(10)
with torch.no_grad():
    for x, y in test_loader:
        pred = model_c(x.to(device)).argmax(1).cpu()
        for c in range(10):
            mask = y == c
            correct[c] += (pred[mask] == c).sum(); total[c] += mask.sum()
plt.figure(figsize=(9, 3))
plt.bar(CLASSES, (correct / total).numpy()); plt.ylim(0, 1); plt.title("per-class test accuracy")
plt.xticks(rotation=30); plt.show()

Cats, dogs and birds are usually the hardest classes: they are visually variable and easily confused with each other. Vehicles and ships are easier because of distinctive shapes and backgrounds.

## 14. Test-time augmentation (bonus)

A cheap trick: average the predictions over the image and its horizontal flip. Since the model was trained to be flip-invariant, this reduces variance in the prediction.

In [ ]:
@torch.no_grad()
def tta_accuracy(model, loader):
    model.eval(); correct = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        probs = F.softmax(model(x), 1) + F.softmax(model(torch.flip(x, dims=[3])), 1)
        correct += (probs.argmax(1) == y).sum().item(); n += y.size(0)
    return correct / n
print(f"test acc without TTA: {test_acc:.4f} | with flip TTA: {tta_accuracy(model_c, test_loader):.4f}")

## Summary

* Natural images expose overfitting: big networks memorise the training set.
* Normalise each channel with **training-set** statistics.
* **Augmentation** encodes invariances you know about; it is usually the single most effective regulariser for vision.
* **BatchNorm** normalises each channel over the batch, has learnable $\gamma, \beta$, and uses running statistics in eval mode. It enables high learning rates.
* **Dropout** randomly zeros activations in training; **weight decay** penalises large weights (use AdamW with Adam).
* **Schedules** (cosine, one-cycle) give fast early progress and precise convergence.
* Change one thing at a time and judge on the validation set.

## Exercises

1. **Ablation.** Train C without BatchNorm (keep everything else). Does it still train at `max_lr=0.1`? What learning rate does it need?
2. **Augmentation strength.** Add `transforms.RandAugment()` or `transforms.RandomErasing(p=0.5)` (after `ToTensor`). Does it help in 15 epochs? In 40?
3. **Small-data regime.** Train A and C on only 5,000 training images. How do the gaps change? Which regulariser matters most when data is scarce?
4. **BatchNorm eval bug.** Evaluate model C with `model_c.train()` instead of `eval()` on test batches of size 8. What happens to accuracy, and why?
5. **Global average pooling.** Replace the Flatten + Linear head by `AdaptiveAvgPool2d(1)` + `Linear(128, 10)` (Notebook 2). Compare parameters and accuracy.
6. **Make it residual.** Add a skip connection inside each `conv_block` (`out = relu(bn(conv(x)) + shortcut(x))`), a mini-ResNet. Read the ResNet paper (He et al., 2016) section 3.1 to understand why identity shortcuts ease optimisation.